# Instrument and Verify a Voice Agent: Make Your Calls Appear, Then Prove It Worked

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/quickstart/instrument-and-verify-voice)

A voice call needs a root span of kind `CONVERSATION`. A trace with only STT, LLM and TTS spans can appear in Traces without appearing as a call. You will send one synthetic support call, check its transcript and span tree with `fi_verify_voice.py`, then open the same trace in the Voice tab.

## Install

Start with a fresh Python 3.12 runtime. Set the named keys in your environment or Colab Secrets, or enter them at the hidden prompts.

Both tracing tracks use the same in-process LiveKit agent. Running both makes two billed synthetic calls through Cartesia, Deepgram, and OpenAI. No audio is saved. Use a fresh notebook directory for the five Python files and the checker.

In [ ]:
%pip install "fi-instrumentation-otel==1.5.1" "livekit-agents[openai,deepgram,cartesia]==1.7.1"

In [ ]:
%pip install opentelemetry-sdk opentelemetry-exporter-otlp-proto-http "livekit-agents[openai,deepgram,cartesia]==1.7.1"

Download the checker and verify its digest before running it.

In [ ]:
%%bash
set -e
curl -fsSL https://docs.futureagi.com/fi_verify_voice.py -o fi_verify_voice.py
python -c 'import hashlib, pathlib; assert hashlib.sha256(pathlib.Path("fi_verify_voice.py").read_bytes()).hexdigest() == "9e487b4e8eb00c1adfb57e2cfdda182005cb8f99d85e909e6531d7730380be2f", "unexpected checker digest"'
printf '%s\n' '.fi_verify/' >> .gitignore

In [ ]:
import os
from getpass import getpass

def load_secret(name):
    if os.environ.get(name):
        return
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except (ImportError, KeyError, RuntimeError):
        value = None
    except Exception as exc:
        if type(exc).__name__ not in {"SecretNotFoundError", "NotebookAccessError"}:
            raise
        value = None
    value = value or getpass(f"{name}: ")
    if not value:
        raise ValueError(f"{name} is required")
    os.environ[name] = value

for name in ['FI_API_KEY', 'FI_SECRET_KEY', 'OPENAI_API_KEY', 'DEEPGRAM_API_KEY', 'CARTESIA_API_KEY']:
    load_secret(name)

os.environ["FI_PROJECT_NAME"] = "ridgeline-voice-support"
os.environ.setdefault("FI_TRACES_URL", "https://api.futureagi.com")
os.environ["FI_ENDPOINT"] = os.environ["FI_TRACES_URL"].rstrip("/") + "/tracer/v1/traces"
os.environ["FI_VOICE_NO_RECORDING"] = "1"

## Check your collector route

In [ ]:
%%bash
set -e
python fi_verify_voice.py preflight

## Configure one tracing provider

In [ ]:
%%writefile setup_sdk.py
import os

from fi_instrumentation import register
from fi_instrumentation.fi_types import ProjectType

provider = register(project_name=os.environ["FI_PROJECT_NAME"],
                    project_type=ProjectType.OBSERVE, verbose=False)
tracer = provider.get_tracer("voice-agent")

In [ ]:
%%writefile setup_otel.py
import os

from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

provider = TracerProvider(resource=Resource.create({
    "project_name": os.environ["FI_PROJECT_NAME"], "project_type": "observe",
}))
provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter(
    endpoint=os.environ["FI_ENDPOINT"],
    headers={"X-Api-Key": os.environ["FI_API_KEY"],
             "X-Secret-Key": os.environ["FI_SECRET_KEY"]},
)))
tracer = provider.get_tracer("voice-agent")

## Give each call one root and a caller

In [ ]:
%%writefile call_scope.py
from contextlib import contextmanager
from opentelemetry.context import Context


@contextmanager
def conversation(tracer, session_id, user_id):
    with tracer.start_as_current_span("voice.call", context=Context(), attributes={
        "fi.span.kind": "CONVERSATION", "session.id": session_id, "user.id": user_id,
    }) as span:
        yield span

## Write what the Voice tab reads

In [ ]:
%%writefile voice_spans.py
import json
from uuid import uuid4


def finish(span, turns, elapsed):
    # Each turn is (role, text, audio duration in seconds).
    rendered = [{"id": uuid4().hex, "role": role, "content": text, "duration": duration}
                for role, text, duration in turns]
    messages = [{"role": role, "content": text} for role, text, _ in turns]
    span.set_attribute("fi.conversation.transcript", json.dumps(rendered))
    span.set_attribute("conversation.transcript", json.dumps(messages))
    for i, (role, text, _) in enumerate(turns):
        span.set_attribute(f"conversation.transcript.{i}.message.role", role)
        span.set_attribute(f"conversation.transcript.{i}.message.content", text)
    caller = sum(duration for role, _, duration in turns if role == "user")
    agent = sum(duration for role, _, duration in turns if role == "assistant")
    span.set_attributes({"call.duration": elapsed, "call.total_turns": len(turns),
                         "call.talk_ratio": agent / caller, "call.status": "completed",
                         "gen_ai.system": "livekit", "input.value": turns[0][1],
                         "output.value": turns[-1][1]})

## Run a real voice exchange

In [ ]:
%%writefile agent.py
import asyncio
import sys
import time
from uuid import uuid4

import aiohttp
from livekit.agents import Agent, AgentSession
from livekit.plugins import cartesia, deepgram, openai

import fi_verify_voice
from call_scope import conversation
from voice_spans import finish

if sys.argv[1:] == ["sdk"]:
    from setup_sdk import provider, tracer
elif sys.argv[1:] == ["otel"]:
    from setup_otel import provider, tracer
else:
    raise SystemExit("Choose a tracing track: sdk or otel")


async def main():
    fi_verify_voice.attach(provider)
    model = "gpt-4o-mini"
    async with (aiohttp.ClientSession() as http,
                cartesia.TTS(http_session=http) as tts,
                deepgram.STT(http_session=http) as stt):
        with conversation(tracer, f"call-{uuid4().hex}", "shopper-demo") as call:
            started = time.monotonic()
            session = AgentSession(llm=openai.LLM(model=model), stt=stt)
            await session.start(agent=Agent(instructions=(
                "You work at an outdoor gear shop. Unused tents can be returned within "
                "30 days. Answer the caller in one short sentence."
            )), record=False)
            try:
                caller_audio = await tts.synthesize("Can I return an unused tent?").collect()
                event = await stt.recognize(caller_audio)
                question = event.alternatives[0].text.strip()
                if not question:
                    raise ValueError("The caller's speech produced no transcript")
                with tracer.start_as_current_span("support.reply", attributes={
                    "fi.span.kind": "LLM", "gen_ai.request.model": model, "input.value": question,
                }) as llm_span:
                    await session.run(user_input=question, input_modality="text")
                    reply = next(m.text_content for m in reversed(session.history.items)
                                 if getattr(m, "role", None) == "assistant" and m.text_content)
                    llm_span.set_attribute("output.value", reply)
                reply_audio = await tts.synthesize(reply).collect()
                turns = [("user", question, caller_audio.duration),
                         ("assistant", reply, reply_audio.duration)]
                finish(call, turns, time.monotonic() - started)
                trace_id = f"{call.get_span_context().trace_id:032x}"
            finally:
                await session.aclose()
    provider.force_flush()
    print(f"Trace ID: {trace_id}")


if __name__ == "__main__":
    asyncio.run(main())

## Check the export and find this call in the app

Check each tracing track immediately after its call. Find each printed trace ID under **Voice** in the `ridgeline-voice-support` project.

In [ ]:
%%bash
set -e
python agent.py sdk
python fi_verify_voice.py check

In [ ]:
%%bash
set -e
python agent.py otel
python fi_verify_voice.py check